In [9]:
import geopandas as gpd
import folium
import branca.colormap as cm
import os

print("Loading vulnerability data...")
grid_path = "data/Islamabad_Vulnerability_Grid.gpkg"
grid_gdf = gpd.read_file(grid_path)

# 1. THE TRIAGE FILTER (Expanded to Top 5% for a better visual spread)
print("Filtering for critical risk zones...")
threshold = grid_gdf['final_vulnerability'].quantile(0.95) 
danger_gdf = grid_gdf[grid_gdf['final_vulnerability'] >= threshold].copy()

# 2. PAYLOAD STRIPPING & DATA CLEANUP
print("Cleaning up data for the web...")
columns_to_keep = ['final_vulnerability', 'density_weight', 'risk_weight', 'soil_score', 'geometry']
danger_gdf = danger_gdf[columns_to_keep]

danger_gdf['final_vulnerability'] = danger_gdf['final_vulnerability'].round(2)
danger_gdf['density_weight'] = danger_gdf['density_weight'].round(2)

# WE DELETED THE POINT CONVERSION HERE! KEEPING THE SEXY POLYGONS!

print("Reprojecting to web coordinates...")
danger_gdf = danger_gdf.to_crs("EPSG:4326")

# 3. COLORMAP SETUP
min_val = danger_gdf['final_vulnerability'].min()
max_val = danger_gdf['final_vulnerability'].max()
colormap = cm.LinearColormap(
    colors=['#fd8d3c', '#e31a1c', '#800026'], # Orange to Dark Red
    vmin=min_val, 
    vmax=max_val
)

print("Generating high-performance web map...")
m = folium.Map(
    location=[33.6844, 73.0479], 
    zoom_start=12, 
    tiles="cartodbdark_matter",
    prefer_canvas=True
)

# 4. NEON POLYGON RENDERING WITH DARK MODE TOOLTIPS
print("Drawing neon city blocks...")
folium.GeoJson(
    danger_gdf,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['final_vulnerability']),
        'color': colormap(feature['properties']['final_vulnerability']), # Border matches fill for a glow effect
        'weight': 0.5,         # Very thin border
        'fillOpacity': 0.85    # High opacity for that neon punch
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['final_vulnerability', 'density_weight', 'risk_weight', 'soil_score'],
        aliases=['Total Vulnerability:', 'Density Score:', 'Fault Proximity:', 'Soil Score:'],
        # Dark mode CSS for the popups!
        style=("background-color: #1a1a1a; color: #ffffff; font-family: arial; font-size: 12px; padding: 10px; border: 1px solid #444; border-radius: 5px; box-shadow: 0px 4px 6px rgba(0,0,0,0.6);")
    )
).add_to(m)

colormap.caption = 'Critical Seismic Vulnerability (Top 5%)'
colormap.add_to(m)

# 5. ADD EXPLANATORY INFO BOX
print("Adding custom legend...")
legend_html = '''
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 280px; height: auto; 
    background-color: rgba(20, 20, 20, 0.85); 
    border: 1px solid #444; z-index:9999; font-size:13px;
    padding: 15px; border-radius: 8px; color: white;
    font-family: Arial, sans-serif; box-shadow: 0 4px 8px rgba(0,0,0,0.5);">
    <h4 style="margin-top:0; margin-bottom:10px; color:#ffcc00;">Seismic Risk Factors</h4>
    <p style="margin:0; margin-bottom:8px; font-size:11px; color:#aaa;">
        Overall Vulnerability is calculated by summing three critical factors:
    </p>
    <b>1. Fault Risk (0-3):</b> Proximity to the active Margalla Fault line.<br><br>
    <b>2. Building Density (0-5):</b> Congestion of structures, impacting evacuation and collapse risk.<br><br>
    <b>3. Soil Type (1-2):</b> Ground composition (soft soils amplify seismic waves).<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# 6. FIX LEGEND TEXT COLOR
css_fix = """
<style>
    .leaflet-control svg text { fill: white !important; }
    .leaflet-control.legend { color: white !important; }
</style>
"""
m.get_root().html.add_child(folium.Element(css_fix))

# 7. EXPORT
output_file = "index.html"
m.save(output_file)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Success! Optimized map saved: {output_file} ({file_size_mb:.2f} MB)")

Loading vulnerability data...
Filtering for critical risk zones...
Cleaning up data for the web...
Reprojecting to web coordinates...
Generating high-performance web map...
Drawing neon city blocks...
Adding custom legend...
Success! Optimized map saved: index.html (16.96 MB)
